# Import Libraries

In [ ]:
from sqlalchemy import create_engine, text
import json, pandas as pd
from agents import Agent, Runner, trace, set_default_openai_client, OpenAIChatCompletionsModel, function_tool
from openai import AsyncOpenAI
import os, asyncio
import gradio as gr

# Database Class

In [ ]:
class MSSQLConnector:
    def __init__(self, server, database, username, password, driver="ODBC Driver 17 for SQL Server"):
        self.server = server
        self.database = database
        self.username = username
        self.password = password
        self.driver = driver.replace(" ", "+")
        self.engine = self._create_engine()

    def _create_engine(self):
        connection_string = (
            f"mssql+pyodbc://{self.username}:{self.password}@{self.server}/{self.database}"
            f"?driver=ODBC+Driver+17+for+SQL+Server"
            f"&TrustServerCertificate=yes"
        )
        engine = create_engine(connection_string, fast_executemany=True)
        return engine

    def read_query(self, query: str, params=None) -> pd.DataFrame:
        with self.engine.connect() as conn:
            df = pd.read_sql(query, conn, params=params)
        return df

    def execute_query(self, query: str, params=None):
        with self.engine.begin() as conn:
            result = conn.execute(text(query), params or {})

            if query.strip().lower().startswith("select"):
                result = result.fetchall()
                return result
            
        print("Query executed successfully")

In [ ]:
# !pip install openai openai-agents gradio

# Set Client

In [ ]:
openaiclient = AsyncOpenAI(
    api_key="xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx",
    base_url="https://api.openai.com/v1"
)

# Function Tools

In [ ]:
cached = {}

server = "xxxxxx"
tgt_database = "ext_emp_data"
username = "xxxxxx"
password = "xxxxxx"

mssql = MSSQLConnector(server, database=tgt_database, username=username, password=password)

# Generate metadata
def generate_metadata(schema, table):
    """return metadata for a table"""  

    query = f"""SELECT COLUMN_NAME, DATA_TYPE FROM INFORMATION_SCHEMA.COLUMNS
    WHERE TABLE_SCHEMA = '{schema}' AND TABLE_NAME = '{table}'  """

    df = mssql.read_query(query=query)
    metadata = [(row['COLUMN_NAME'] + " " + row['DATA_TYPE']) for _ , row in df.iterrows()]
    cached[table] = metadata
    return cached

# Get metadata
@function_tool
def get_metadata(schema, table):
    mtstr = ""
    if (table not in cached) or (cached[table] == ''):
        print('Generating metadata...')
        result = generate_metadata(schema, table)
        metadata = cached[table]
    else:
        print('Metadata present in cache...')
        metadata = cached[table]
        for i in metadata:
            mtstr = mtstr+","+i
        return mtstr

# Fetch data    
@function_tool
def fetch_data(query):
    df = mssql.read_query(query).head(10)
    html_table = df.to_html(index=False)
    return html_table

# Execute DDL/DML/TCL
@function_tool
def execute_sql(query):
    try:
        mssql.execute_query(query)
        return "OK"
    except Exception as e:
        return str(e)

# Agent

In [ ]:
instructions = """
You are a SQL Server expert who generates clean optimized code by understanding
user query. You have access to the following tools:
1. get_metadata - to fetch metadata for any table in the database
2. fetch_data - takes SQL as input and return data as HTML table format
3. execute_sql - to execute any DDL or DML in the database

You need to perform the following steps in order:
1. Understand user query.
2. If user want to query table or view data then:
    a) Always ask for schema and table name if not given by user
    b) Use the schema and table name and call the get_metadata function to get metadata of the table.
    c) Generate SQL code based on the metadata and user query.
    d) Take the query as is as input and call the fetch_data.
    e) Always ask the user for clarification if you have doubt.

3. If user asks to run a DDL or DML or TCL like delete or truncate or create table or create role anything:
    a) Generate the query based on user request.
    b) Get back with question if query is not clear to you.
    c) Generate the SQL and ask for user confirmation by showing the query. 
    d) If user confirms then call execute_sql tool to run the SQL.
    e) Always ask the user for clarification if you have doubt.

Return the following outputs in nice Gradio supported HTML:
1. Return the the output of fetch_data in a html format starting answer with "Here is the result:" and then a new line.
2. Return the generated SQL query in a sql ``` fence starting answer with "Here is the SQL:" and then a new line.
3. 1 follow up question related to the table based on the previous user question starting with "Do you want to know"

Your default greetings message will be:
Hi, how can I help you today? I am a SQL Server Expert. I can fetch data from any table, run DDL DML TCL.
"""

sql_gen = Agent(name="SQL Agent", instructions=instructions, tools=[get_metadata, fetch_data, execute_sql], 
                        model=OpenAIChatCompletionsModel(
                        model="gpt-5-nano",
                        openai_client=openaiclient,
                    ))

# Conversation History

In [ ]:
hist = []

async def chat(message, history=None):
    with trace("SQL Chat Flow"):
        conversation = hist + [{'role': 'user', 'content': message}]
        print('conversation:', conversation)
        result = await Runner.run(sql_gen, conversation)
        hist.append({'role': 'user', 'content': message})
        hist.append({'role': 'assistant', 'content': result.final_output})
        print('result:',result.final_output)
        print('hist:',hist)
    return result.final_output

# Gradio UI

In [ ]:
theme = gr.themes.Soft(
    primary_hue="violet",
    secondary_hue="cyan",
).set(
    body_background_fill="#f2f3ed",           
    body_text_color="#030d1d",               
    block_background_fill="#e1ebdfcc",         
    block_border_color="#212f78",          
    block_title_text_color="#2254c0",         
    block_shadow="0 4px 15px rgba(99, 102, 241, 0.3)", 
    block_radius="16px",
    link_text_color="#67679e",            
    input_background_fill="#eef2ff",        
    input_border_color="#8a91d3",             
    button_primary_background_fill="#6f709e",  
    button_primary_text_color="white",
    button_secondary_background_fill="#789599",
    button_secondary_text_color="#0f172a",
)

gr.ChatInterface(
    fn=chat,
    type="messages",
    title="🎨 SQL Assistant",
    description="💬 Ask me anything about your database — I’ll generate SQL and show results beautifully!",
    theme=theme,
    examples=[
        "🔍 Show top 5 records from emp.employee table",
        "📋 Create view emp.vw_emp on emp.employee table",
        "💰 Show top 3 employee by salary"
    ],
).launch()